In [16]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [17]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Размер тренировочных данных: {train_df.shape}")
print(f"Размер тестовых данных: {test_df.shape}")
print(f"\nПервые 5 строк:")
train_df.head()

Размер тренировочных данных: (10800, 4)
Размер тестовых данных: (973, 3)

Первые 5 строк:


,id,title,body,label
0,0,Ok guys I have a dilemma for you,Ok so I’m thinking about asking out this girl ...,0.0
1,1,Update from my cringe overdramatic ass post,56 days ago from today I posted about my then-...,0.0
2,2,"Don't know if this is allowed, I work in youth...",NaN,0.0
3,3,So I think I just ascended to a newer level of...,So I was rebattling the elite four in Pokemon ...,0.0
4,4,My friend got a gf before me,But I have more reddit karma so I think I win 😎,0.0


In [18]:
print("Пропуски в данных:")
print(train_df.isnull().sum())
print("\n" + "="*50)
print("Пропуски в тестовых данных:")
print(test_df.isnull().sum())
train_df.iloc[:, 1] = train_df.iloc[:, 1].fillna('empty')
test_df.iloc[:, 1] = test_df.iloc[:, 1].fillna('empty')

Пропуски в данных:
id          0
title       0
body     2074
label       0
dtype: int64

Пропуски в тестовых данных:
id       0
title    0
body     0
dtype: int64


In [19]:
print("Распределение классов в тренировочных данных:")
class_distribution = train_df.iloc[:, 0].value_counts()
print(class_distribution)
print(f"\nПроцент положительного класса: {class_distribution[1] / len(train_df) * 100:.2f}%")

Распределение классов в тренировочных данных:
id
0        1
1        1
2        1
3        1
4        1
        ..
10795    1
10796    1
10797    1
10798    1
10799    1
Name: count, Length: 10800, dtype: int64

Процент положительного класса: 0.01%


In [20]:
train_df['title'] = train_df['title'].fillna('')
train_df['body'] = train_df['body'].fillna('')
test_df['title'] = test_df['title'].fillna('')
test_df['body'] = test_df['body'].fillna('')

train_df['text'] = train_df['title'] + ' ' + train_df['body']
test_df['text'] = test_df['title'] + ' ' + test_df['body']

y = train_df['label'].astype(int).values

In [21]:
nltk.download('punkt')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
train_texts = train_df['text'].apply(clean_text)
test_texts = test_df['text'].apply(clean_text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\кошька\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\кошька\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.8,
    sublinear_tf=True
)
X_train = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)

print(f"Features: {X_train.shape[1]}")

Features: 15000


In [23]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y, test_size=0.2, random_state=42, stratify=y)

sgd = SGDClassifier(loss='log_loss', max_iter=1000, class_weight='balanced', random_state=42)
sgd.fit(X_tr, y_tr)
y_pred = sgd.predict(X_val)
acc_sgd = accuracy_score(y_val, y_pred)
print(f"SGD: {acc_sgd:.4f}")

svc = LinearSVC(max_iter=1000, class_weight='balanced', random_state=42, dual=False)
svc.fit(X_tr, y_tr)
y_pred = svc.predict(X_val)
acc_svc = accuracy_score(y_val, y_pred)
print(f"SVC: {acc_svc:.4f}")

lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, solver='saga')
lr.fit(X_tr, y_tr)
y_pred = lr.predict(X_val)
acc_lr = accuracy_score(y_val, y_pred)
print(f"LR: {acc_lr:.4f}")

SGD: 0.9111
SVC: 0.9218
LR: 0.9097


In [24]:
ensemble = VotingClassifier([
    ('sgd', SGDClassifier(loss='log_loss', max_iter=1000, class_weight='balanced', random_state=42)),
    ('svc', LinearSVC(max_iter=1000, class_weight='balanced', random_state=42, dual=False)),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42, solver='saga'))
], voting='hard')
ensemble.fit(X_tr, y_tr)
y_pred = ensemble.predict(X_val)
acc_ens = accuracy_score(y_val, y_pred)
print(f"Ensemble: {acc_ens:.4f}")

models = {'sgd': (sgd, acc_sgd), 'svc': (svc, acc_svc), 'lr': (lr, acc_lr), 'ensemble': (ensemble, acc_ens)}
best_name = max(models, key=lambda x: models[x][1])
best_model = models[best_name][0]
print(f"Best: {best_name} ({models[best_name][1]:.4f})")

Ensemble: 0.9106
Best: svc (0.9218)


In [25]:
best_model.fit(X_train, y)
predictions = best_model.predict(X_test)
print(f"0:{(predictions==0).sum()}, 1:{(predictions==1).sum()}")


submission = pd.DataFrame({
    'id': test_df['id'],
    'label': predictions
})
submission.to_csv('submission.csv', index=False)
print("saved")

0:735, 1:238
saved
